* 기본 설정

In [1]:
import os
import pathlib

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day04" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w3" / "day04"

print("프로젝트 루트  :", ROOT)

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [2]:
import anthropic
import sys

print("anthropic 버전 : ", anthropic.__version__)
print("지금의 파이썬 커널 버전 ", sys.executable)

anthropic 버전 :  1.6.0
지금의 파이썬 커널 버전  /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/.venv/bin/python


In [3]:
import sys

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

In [4]:
from app.core.config import get_settings, mask

settings = get_settings()
print('모드        :', settings.app_mode)
print('모델        :', settings.llm_model)

if settings.anthropic_api_key is None:
    print('API 키      : (없음) — .env 의 ANTHROPIC_API_KEY 가 비어 있습니다')
else:
    print('API 키      :', mask(settings.anthropic_api_key.get_secret_value()))


모드        : mock
모델        : claude-haiku-4-5
API 키      : sk-ant-a...(108자)


In [5]:
import inspect

from anthropic.resources.messages import Messages

# Message.creat : API 한테 보내줄 메시지 작성하는 기능
params = inspect.signature(Messages.create).parameters
# vlftn vkfkalxj whghl
required = [name for name, p in params.items() if p.default is inspect.Parameter.empty and name != "self"]
print(required)

QUESTION = "제주도 출장 숙박비 한도가 얼마인가요?"

try:
    Messages.create(
        None, # client가 와야하는 자리
        model="clauded-haiku-4-5",
        # max_tokens 입력 안하면 TypeError
        messages = [
            {"role" : "User", "content" : QUESTION}
        ],
    )
except TypeError as exc:
    print(type(exc).__name__)
    print(exc)

['max_tokens', 'messages', 'model']
TypeError
Missing required arguments; Expected either ('max_tokens', 'messages' and 'model') or ('max_tokens', 'messages', 'model' and 'stream') arguments to be given


* 금액 계산기

In [7]:
PRICING = {'input': 1.0, 'cache_write': 1.25, 'cache_read': 0.1, 'output': 5.0}
USD_KRW = 1400.0

def cost_krw(input_tok: int, output_tok: int) -> float:
    usd = (input_tok * PRICING['input'] + output_tok * PRICING['output']) / 1000000
    return round(usd * USD_KRW, 1)

total_input = 69
output_tok = 146

(WED_INPUT, WED_OUTPUT) = (800, 400)
wed = cost_krw(WED_INPUT, WED_OUTPUT) # 가짜 금액
today = cost_krw(total_input, output_tok)
print(f'어제 어림한 값 : {wed } 원   (입력 {WED_INPUT} · 출력 {WED_OUTPUT} 토큰)')
print(f'오늘 실제 호출     : {today} 원   (입력 {total_input} · 출력 {output_tok} 토큰)')
print(f'차이               : {round(wed - today, 1)} 원')

어제 어림한 값 : 3.9 원   (입력 800 · 출력 400 토큰)
오늘 실제 호출     : 1.1 원   (입력 69 · 출력 146 토큰)
차이               : 2.8 원


* 포트 확인

In [8]:
# 파이썬은 한번 훑은 폴더의 파일 목록을 기억해둔다.
# 노트북 처럼 돌면서 파일을 새로 만드는 자리에서는, 방금 만든 파일을 못 볼 수 있다.
# 아래 invalidate_caches()로 그 기억을 비워서 다시 훑게 만들기.
import importlib
importlib.invalidate_caches()


from app.integrations.ports import LLMPort, LLMResult

result = LLMResult(text="1박 70,000원 이내", model="claude-haiku-4-5")
print(result)
print("기본값 확인")
print("input_token: ", result.input_tok)
print("extras: ", result.extras)
print("cost_krw: ", result.cost_krw)
print("cost_krw: ", LLMPort.__name__)

LLMResult(text='1박 70,000원 이내', model='claude-haiku-4-5', input_tok=0, cache_tok=0, output_tok=0, cost_krw=0.0, latency_ms=0, extras={})
기본값 확인
input_token:  0
extras:  {}
cost_krw:  0.0
cost_krw:  LLMPort


In [9]:
import importlib
importlib.invalidate_caches() 

from app.integrations.llm_claude import USD_KRW, estimate_cost_krw

input_tok = 62
output_tok = 118

print("환율 : ", USD_KRW)
print("계산한 비용 : ", estimate_cost_krw(input_tok, output_tok), "원")

환율 :  1400.0
계산한 비용 :  0.9 원


In [10]:
import importlib
import inspect
from app.integrations import llm_claude

llm_claude = importlib.reload(llm_claude)

lines = inspect.getsource(llm_claude.ClaudeLLM.__init__).splitlines()

(shown, inside_doc) = ([], False)

for line in lines:
    if line.strip().startswith('"""'):
        inside_doc = not inside_doc
        continue
    if not inside_doc:
        shown.append(line)
        
print('── ClaudeLLM.__init__ 소스 (docstring 은 접었다) ──')
print('\n'.join(shown))
print('answer 있는가:', hasattr(llm_claude.ClaudeLLM, 'answer'))


── ClaudeLLM.__init__ 소스 (docstring 은 접었다) ──
    def __init__(self) -> None:
        try:
            from anthropic import Anthropic
        except ImportError as exc:
            raise ExternalServiceError(
                "anthropic 패키지가 설치 되어 있지 않습니다."
            ) from exc

        settings = get_settings()
        key = settings.anthropic_api_key
        if key is None:
            raise ExternalServiceError("ANTHROPIC_API_KEY 가 비어있습니다.")
        self._client = Anthropic(api_key=key.get_secret_value())
        self.__model = settings.llm_model
answer 있는가: False


In [11]:
import importlib
import inspect
from app.integrations import llm_claude, ports
llm_claude = importlib.reload(llm_claude)
sig_port = inspect.signature(ports.LLMPort.answer, eval_str=True)
sig_impl = inspect.signature(llm_claude.ClaudeLLM.answer, eval_str=True)
print('LLMPort.answer   :', sig_port)
print('ClaudeLLM.answer :', sig_impl)
print()
print('_load_prompt 있는가   :', hasattr(llm_claude, '_load_prompt'), '  ← 노트북 02 에서 만든다')
print('_context_block 있는가 :', hasattr(llm_claude, '_context_block'), '  ← 노트북 03 에서 만든다')

LLMPort.answer   : (self, *, question: str, contexts: list[dict], user: dict) -> app.integrations.ports.LLMResult
ClaudeLLM.answer : (self, *, question: str, contexts: list[dict], user: dict) -> app.integrations.ports.LLMResult

_load_prompt 있는가   : False   ← 노트북 02 에서 만든다
_context_block 있는가 : False   ← 노트북 03 에서 만든다
